<a href="https://colab.research.google.com/github/inspire2029-sudo/phantoms-ai-week2/blob/main/W2D4_Automation_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
!pip install -q requests schedule python-dotenv


In [44]:
import requests
import sqlite3
import json
import schedule
import time
from datetime import datetime

In [45]:
from google.colab import userdata

API_KEY = userdata.get("OPENWEATHER_API_KEY")
if not API_KEY:
    raise ValueError("Add OPENWEATHER_API_KEY to Google Colab Secrets before running this cell.")

API_URL = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "q": "Alexandria,EG",
    "appid": API_KEY,
    "units": "metric"
}

response = requests.get(API_URL, params=params, timeout=10)

print("Status Code:", response.status_code)
response.raise_for_status()
data = response.json()
print("Weather data fetched successfully")


In [46]:
import sqlite3
import json
from datetime import datetime

connection = sqlite3.connect("store.db")
cursor = connection.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS api_logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        api_data TEXT NOT NULL,
        timestamp TEXT NOT NULL
    )
""")

connection.commit()
connection.close()

print("Database and table are ready")

### `store.db  في  API  هنخزن الـ `

In [47]:
connection = sqlite3.connect("store.db")
cursor = connection.cursor()

cursor.execute(
    """
    INSERT INTO api_logs (api_data, timestamp)
    VALUES (?, ?)
    """,
    (
        json.dumps(data, ensure_ascii=False),
        datetime.now().isoformat()
    )
)

connection.commit()
connection.close()

print("API data stored successfully")

### `بتأكد انها دخلت فعلا`

In [48]:
connection = sqlite3.connect("store.db")
cursor = connection.cursor()

cursor.execute("""
    SELECT id, api_data, timestamp
    FROM api_logs
    ORDER BY id DESC
    LIMIT 1
""")

row = cursor.fetchone()

connection.close()

print("Last stored record:")
print(row)

In [49]:
from google.colab import userdata

WEBHOOK_URL = userdata.get("WEBHOOK_URL")
if not WEBHOOK_URL:
    raise ValueError("Add WEBHOOK_URL to Google Colab Secrets before running this cell.")

webhook_response = requests.post(
    WEBHOOK_URL,
    json=data,
    timeout=10
)
webhook_response.raise_for_status()

print("Webhook Status Code:", webhook_response.status_code)
print("Webhook Response:", webhook_response.text)


In [50]:
webhook_response = requests.post(
    WEBHOOK_URL,
    json=data,
    timeout=10
)
webhook_response.raise_for_status()

print("Webhook Status Code:", webhook_response.status_code)
print("Webhook Response:", webhook_response.json())


### `كدا الاوثانتيكيشن تمام + سحب البيانات + تخزين البيانات + ارساله لـ الويب هوك  `
### `وباقي الـ اوتوميشن`

### `run_pipeline() الـ `

In [51]:
def run_pipeline(cycle_number):
    print(f"\n--- بدء الدورة رقم {cycle_number} ---")

    # 1.API هنجيب الداتا من الـ
    response = requests.get(
        API_URL,
        params=params,
        timeout=10
    )

    if response.status_code != 200:
        print("فشل الاتصال بالـ API")
        print("Status Code:", response.status_code)
        return False

    data = response.json()
    print(" تم احضار الداتا من API")

    # 2. تخزين البيانات في Database
    connection = sqlite3.connect("store.db")
    cursor = connection.cursor()

    cursor.execute(
        """
        INSERT INTO api_logs (api_data, timestamp)
        VALUES (?, ?)
        """,
        (
            json.dumps(data, ensure_ascii=False),
            datetime.now().isoformat()
        )
    )

    connection.commit()
    connection.close()

    print("تم تخزين البيانات في Database")

    # 3. إرسال نفس البيانات إلى Webhook
    webhook_response = requests.post(
        WEBHOOK_URL,
        json=data,
        timeout=10
    )

    if webhook_response.status_code != 200:
        print("فشل إرسال البيانات إلى Webhook")
        print("Status Code:", webhook_response.status_code)
        print("Response:", webhook_response.text)
        return False

    print("تم إرسال البيانات إلى Webhook")

    print(f"تم تنفيذ الدورة رقم {cycle_number} بنجاح")

    return True

In [52]:
run_pipeline(1)

In [53]:
import time

for cycle_number in range(1, 4):
    success = run_pipeline(cycle_number)

    if cycle_number < 3:
        print("انتظار 30 ثانية قبل الدورة التالية...")
        time.sleep(30)

### `اخر تحقق بشوف بيه لو ال 3 دورات اتخزنوا فعلا`

In [54]:
connection = sqlite3.connect("store.db")
cursor = connection.cursor()

cursor.execute("""
    SELECT COUNT(*)
    FROM api_logs
""")

count = cursor.fetchone()[0]

connection.close()

print("Total API records stored:", count)

### `المف النهائى`

In [55]:
%%writefile W2D4_Automation_Pipeline.py

import os
import requests
import sqlite3
import json
import time
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENWEATHER_API_KEY")
WEBHOOK_URL = os.getenv("WEBHOOK_URL")

if not API_KEY:
    raise RuntimeError("OPENWEATHER_API_KEY is not set")
if not WEBHOOK_URL:
    raise RuntimeError("WEBHOOK_URL is not set")

API_URL = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "q": "Alexandria,EG",
    "appid": API_KEY,
    "units": "metric"
}


def init_database():
    connection = sqlite3.connect("store.db")
    cursor = connection.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS api_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            api_data TEXT NOT NULL,
            timestamp TEXT NOT NULL
        )
    """)
    connection.commit()
    connection.close()


def run_pipeline(cycle_number):
    print(f"\\n--- Starting cycle {cycle_number} ---")

    response = requests.get(API_URL, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()
    print("API data fetched successfully")

    connection = sqlite3.connect("store.db")
    cursor = connection.cursor()
    cursor.execute(
        """
        INSERT INTO api_logs (api_data, timestamp)
        VALUES (?, ?)
        """,
        (
            json.dumps(data, ensure_ascii=False),
            datetime.now(timezone.utc).isoformat(),
        ),
    )
    connection.commit()
    connection.close()
    print("API data stored successfully")

    webhook_response = requests.post(WEBHOOK_URL, json=data, timeout=10)
    webhook_response.raise_for_status()
    print("Data sent to webhook successfully")
    print(f"Cycle {cycle_number} completed successfully")


if __name__ == "__main__":
    init_database()
    for cycle_number in range(1, 4):
        run_pipeline(cycle_number)
        if cycle_number < 3:
            time.sleep(30)


### `بتأكد ان الملف موجود`

In [56]:
import os

print(os.path.exists("W2D4_Automation_Pipeline.py"))

In [57]:
with open("W2D4_Automation_Pipeline.py", "r", encoding="utf-8") as file:
    content = file.read()

print("YOUR_OPENWEATHER_API_KEY" in content)

In [58]:
with open("W2D4_Automation_Pipeline.py", "r", encoding="utf-8") as file:
    print(file.read())

In [59]:
from google.colab import userdata

API_KEY = userdata.get("OPENWEATHER_API_KEY")

if not API_KEY:
    raise ValueError("Add OPENWEATHER_API_KEY to Google Colab Secrets.")

print("API Key loaded:", True)


In [60]:
from google.colab import userdata

API_KEY = userdata.get("OPENWEATHER_API_KEY")

if not API_KEY:
    raise ValueError("Add OPENWEATHER_API_KEY to Google Colab Secrets.")

print("API Key loaded:", True)


In [61]:
API_URL = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "q": "Alexandria,EG",
    "appid": API_KEY,
    "units": "metric"
}

print("Parameters updated")